In [3]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# ===== Stateクラスの定義 =====
class State(TypedDict):
    messages: Annotated[list, add_messages]

# ===== グラフの構築 =====
def build_graph(model_name):
    # LLM
    llm = ChatOpenAI(model=model_name, temperature=0)

    # 検索ツール
    search = TavilySearchResults(max_results=3)

    tools = [search]

    # LLMにツールをバインド
    llm_with_tools = llm.bind_tools(tools)

    # ノード定義
    def chatbot(state: State):
        return {
            "messages": [
                llm_with_tools.invoke(state["messages"])
            ]
        }

    tool_node = ToolNode(tools)

    # グラフ構築
    graph = StateGraph(State)

    graph.add_node("chatbot", chatbot)
    graph.add_node("tools", tool_node)

    # エントリーポイント
    graph.set_entry_point("chatbot")

    # 分岐（ツール呼び出しかどうか）
    graph.add_conditional_edges(
        "chatbot",
        tools_condition
    )

    # ツール → 再びLLM
    graph.add_edge("tools", "chatbot")

    # メモリ（会話保持）
    memory = MemorySaver()

    return graph.compile(checkpointer=memory)

# ===== グラフ実行関数 =====
def stream_graph_updates(graph: StateGraph, user_input: str):
    events = graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values"
    )

    for event in events:
        print(event["messages"][-1].content, flush=True)

# ===== メイン実行ロジック =====
# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini" 

# グラフの作成
graph = build_graph(MODEL_NAME)

# メインループ
while True:
    user_input = input("質問:")
    if user_input.strip()=="":
        print("ありがとうございました!")
        break
    stream_graph_updates(graph, user_input)

こんにちは！
こんにちは！今日はどんなことをお手伝いできますか？
1たす2は？
1たす2は3です。何か他に知りたいことがありますか？
台湾観光について検索結果を教えて

[{"url": "https://www.agu.ac.jp/pdf/graduate/thesis/letters1/35-3.pdf", "content": "「熱帯産業調査会」が設置さ れた後、 「観光台湾」というスローガンが出てくるようになる。それは以下の新聞記事 からも示されている。 台湾は四面環海、常夏でかつ常春、百花競ひ咲き山野また常緑の美装を讃え ている上に、ここに住むものの風俗慣習は内地人、本島人、高砂族とみなそれ ぞれの異色を持ってゐる、これだけでも「観光台湾」の姿は多彩である上に、 熱帯と温帯とを半々に持つ台湾は内地同様の植物も一通り存在するほか椰子、 マンゴー、パパイヤ、パイナップル、バナナなどの南国植物による情趣も極め て多様に満ちてをり…台湾の観光価値、観光資源は大体前にも述べた通り頗る 豊富であって、堂々と対外宣伝を行って、外客を誘致する価値十分であるに拘 47 らず、この天奥の資源もまだ開発されない憾みがあるすなはちこれらの資源を 開発して観光設備を充実するとともに大いに対外宣伝に努力するの要ある… 台湾としての観光事業の対策…第一に観光組織の整備統制である…第二は観 光宣伝の強化徹底である、第三は観光設備の充実である（ 「観光台湾の問題、 施設充実と宣伝徹底で、世界に呼びかけよう」 『大阪朝日新聞－台湾版』昭和 11（1936）年9 月8 日） 。 この新聞記事によれば、1930 年代当時、台湾の観光資源は豊富であり、対外宣伝を 行って外客を誘致するには十分、とされる。しかし、これらの観光資源を開発するた めには、組織の整備統制と設備を充実させるには不十分で、台湾における観光宣伝を 徹底的に強化しなければならない、ことが分かる。また、この時の日本にとって、台 湾の観光事業を発展させることは、国際的にアピールするだけでなく、台灣の経済状 況を改善・緩和させる手段の一環でもあった、と考えられる。 第三節 台湾人の日本旅行と視察団について １．台湾人日本旅行 林淑慧「借鏡他山之石： [...] 『大阪朝日新聞－台湾版』1936 年９月８日） 。 （下線は筆者による） 